# Comparing DAC, MAC, and RBAC on the Same Requests


## Teaching goal

This notebook puts the three toy models side by side. The same request may be allowed in one model
and denied in another because each model asks a different policy question.


In [ ]:
from dataclasses import dataclass
from pprint import pprint


@dataclass(frozen=True)
class Request:
    user: str
    operation: str
    resource: str
    context: dict


users = {
    "dr_rossi": {"name": "Dr. Rossi", "department": "cardiology"},
    "nurse_amina": {"name": "Nurse Amina", "department": "ward-a"},
    "lab_tech": {"name": "Lab Technician", "department": "laboratory"},
    "billing_clerk": {"name": "Billing Clerk", "department": "billing"},
    "privacy_auditor": {"name": "Privacy Auditor", "department": "compliance"},
}

resources = {
    "ehr_note_42": {"type": "ehr_note", "patient": "patient-42", "department": "cardiology"},
    "lab_result_42": {"type": "lab_result", "patient": "patient-42", "department": "laboratory"},
    "billing_record_42": {"type": "billing_record", "patient": "patient-42", "department": "billing"},
    "audit_log": {"type": "audit_log", "patient": None, "department": "compliance"},
}

requests = [
    Request("dr_rossi", "read", "ehr_note_42", {"assigned_patient": True, "emergency": False}),
    Request("nurse_amina", "write", "ehr_note_42", {"assigned_patient": True, "emergency": False}),
    Request("lab_tech", "write", "lab_result_42", {"assigned_patient": False, "emergency": False}),
    Request("billing_clerk", "read", "ehr_note_42", {"assigned_patient": False, "emergency": False}),
    Request("privacy_auditor", "read", "audit_log", {"assigned_patient": False, "emergency": False}),
]


In [ ]:
# Minimal DAC policy.
dac_matrix = {
    "dr_rossi": {"ehr_note_42": {"read", "write"}},
    "nurse_amina": {"ehr_note_42": {"read", "write"}},
    "lab_tech": {"lab_result_42": {"read", "write"}},
    "billing_clerk": {"billing_record_42": {"read", "write"}},
    "privacy_auditor": {"audit_log": {"read"}},
}

# Minimal MAC policy.
ranks = {"internal": 1, "confidential": 2, "restricted": 3}
clearances = {
    "dr_rossi": {"rank": "restricted", "compartments": {"cardiology"}},
    "nurse_amina": {"rank": "confidential", "compartments": {"ward-a", "cardiology"}},
    "lab_tech": {"rank": "confidential", "compartments": {"laboratory"}},
    "billing_clerk": {"rank": "internal", "compartments": {"billing"}},
    "privacy_auditor": {"rank": "restricted", "compartments": {"compliance", "cardiology"}},
}
labels = {
    "ehr_note_42": {"rank": "confidential", "compartments": {"cardiology"}},
    "lab_result_42": {"rank": "confidential", "compartments": {"laboratory"}},
    "billing_record_42": {"rank": "internal", "compartments": {"billing"}},
    "audit_log": {"rank": "restricted", "compartments": {"compliance"}},
}

# Minimal RBAC policy.
user_roles = {
    "dr_rossi": {"attending_physician"},
    "nurse_amina": {"ward_nurse"},
    "lab_tech": {"lab_technician"},
    "billing_clerk": {"billing_staff"},
    "privacy_auditor": {"auditor"},
}
role_permissions = {
    "attending_physician": {("read", "ehr_note"), ("write", "ehr_note"), ("read", "lab_result")},
    "ward_nurse": {("read", "ehr_note"), ("write", "ehr_note"), ("read", "lab_result")},
    "lab_technician": {("read", "lab_result"), ("write", "lab_result")},
    "billing_staff": {("read", "billing_record"), ("write", "billing_record")},
    "auditor": {("read", "audit_log")},
}


In [ ]:
def dac(request):
    return request.operation in dac_matrix.get(request.user, {}).get(request.resource, set())


def mac(request):
    # This toy MAC function allows reads only, to keep the example focused.
    if request.operation != "read":
        return False
    clearance = clearances[request.user]
    label = labels[request.resource]
    rank_ok = ranks[clearance["rank"]] >= ranks[label["rank"]]
    compartments_ok = label["compartments"].issubset(clearance["compartments"])
    return rank_ok and compartments_ok


def rbac(request):
    resource_type = resources[request.resource]["type"]
    needed_permission = (request.operation, resource_type)
    return any(
        needed_permission in role_permissions[role]
        for role in user_roles.get(request.user, set())
    )


for request in requests:
    print(f"{request.user:16} {request.operation:5} {request.resource:18}",
          "DAC=", dac(request),
          "MAC=", mac(request),
          "RBAC=", rbac(request))


In [ ]:
# Try changing one policy and rerun the comparison.
# Example: give billing staff read access to EHR notes in RBAC.
role_permissions["billing_staff"].add(("read", "ehr_note"))

request = Request("billing_clerk", "read", "ehr_note_42", {"assigned_patient": False})
print("After RBAC policy change:")
print("DAC=", dac(request), "MAC=", mac(request), "RBAC=", rbac(request))


## What to notice

The model changes the question:

- DAC asks: did this identity receive this right on this object?
- MAC asks: does the subject clearance dominate the object label?
- RBAC asks: does an active/assigned role grant this operation?

Real systems provide the enforcement mechanisms. Administrators and security teams decide the
policies, review them, and align them with clinical workflow and legal duties.
